# Deep Learning 011 — Building an ANN with Keras: Customer Churn

Companion notebook to the lesson. The first end-to-end network: a bank has 10,000
customers and wants to know which ones are about to leave.

The modelling is the easy part. **Everything that decides whether this works happens before
`model.fit`** — the class balance, the categorical encoding, and the feature scaling — so
that is where most of this notebook is.

> **TensorFlow is optional here.** Every preprocessing and diagnostic cell runs on pandas
> and scikit-learn alone. The Keras cells are marked and will run for you if you have
> TensorFlow installed; a scikit-learn `MLPClassifier` with the same architecture is
> included so the whole notebook produces numbers either way.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.width", 120)
df = pd.read_csv("../data/Churn_Modelling.csv")
print(df.shape)
df.head()

## The single most important number in this dataset

In [ ]:
counts = df["Exited"].value_counts()
print(counts)
print(f"\nstayed {counts[0] / len(df):.1%}   left {counts[1] / len(df):.1%}")
print(f"\nA model that predicts 'nobody ever leaves' scores {counts[0] / len(df):.1%} accuracy")
print("and has learned nothing whatsoever.")

**79.6% is the number to beat, not 50%.** Write it down before training anything. Accuracy
on an imbalanced dataset is a misleading metric, and a network that reports 79.6% after
training has almost certainly collapsed to the majority class.

## Which columns are inputs?

Three columns are identifiers, not features. Feeding them to a network invites it to
memorise row numbers.

In [ ]:
drop = ["RowNumber", "CustomerId", "Surname"]
for c in drop:
    print(f"{c:>12}: {df[c].nunique()} unique values out of {len(df)}")
print("\nnothing generalisable in any of them")

X = df.drop(columns=drop + ["Exited"])
y = df["Exited"].values
print("\nfeature dtypes:")
print(X.dtypes.to_string())

## Categorical columns — and why `drop_first=True`

`Geography` and `Gender` are text. A network multiplies its inputs by weights, so they have
to become numbers — and **not** by mapping France→0, Germany→1, Spain→2, which would assert
that Spain is twice Germany.

One-hot encoding is the right answer, with one refinement: the last column is redundant.
If a customer is not in France and not in Germany, they are in Spain — the third column
carries no information the first two lack.

In [ ]:
print("Geography:", df["Geography"].unique().tolist())
print("Gender   :", df["Gender"].unique().tolist())

full = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=False)
lean = pd.get_dummies(X, columns=["Geography", "Gender"], drop_first=True)
print(f"\nwithout drop_first: {full.shape[1]} columns  {[c for c in full.columns if '_' in c]}")
print(f"with    drop_first: {lean.shape[1]} columns  {[c for c in lean.columns if '_' in c]}")

# the dropped column is recoverable, which is exactly why it is redundant
recovered = 1 - lean["Geography_Germany"] - lean["Geography_Spain"]
assert (recovered == full["Geography_France"]).all()
print("\nGeography_France == 1 - Germany - Spain, exactly. It was never needed.")
X = lean.astype(float)

## Scaling is not optional here

Look at the ranges. `Balance` runs to a quarter of a million; `HasCrCard` is 0 or 1. Both
get multiplied by a weight initialised from the same small distribution, so before any
training happens the balance term is a hundred thousand times louder.

In [ ]:
spans = (X.max() - X.min()).sort_values(ascending=False)
print(spans.to_string())
print(f"\nratio between the widest and narrowest span: {spans.iloc[0] / spans.iloc[-1]:,.0f}x")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X.values, y, test_size=0.2, random_state=0, stratify=y)

scaler = StandardScaler().fit(X_train)          # fit on TRAIN only
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)

print(f"train {X_train_s.shape}, test {X_test_s.shape}")
print(f"train mean {X_train_s.mean():.2e}, std {X_train_s.std():.3f}")
print(f"test  mean {X_test_s.mean():+.4f}, std {X_test_s.std():.3f}   <- not exactly 0/1, and correct")

The test set's mean is *not* zero, and that is the point: the scaler was fitted on the
training data only. Fitting it on everything leaks information about the test set into
training, and the score you get back is then a slightly optimistic lie.

## The architecture

| Layer | Nodes | Activation | Why |
|---|---|---|---|
| input | 11 | — | one per feature after encoding |
| hidden 1 | 11 | ReLU | a starting point, not a law |
| hidden 2 | 11 | ReLU | |
| output | **1** | **sigmoid** | binary classification, fixed by the task |

The output layer is not a choice. One node, sigmoid, binary cross-entropy — that trio is
determined by "the answer is one probability between 0 and 1".

In [ ]:
# --- Keras version. Runs if you have TensorFlow installed. ---
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Input(shape=(X_train_s.shape[1],)),
    keras.layers.Dense(11, activation="relu"),
    keras.layers.Dense(11, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

history = model.fit(X_train_s, y_train, epochs=50, batch_size=32,
                    validation_split=0.2, verbose=0)
loss_, acc = model.evaluate(X_test_s, y_test, verbose=0)
print(f"test accuracy {acc:.4f}   (baseline to beat: 0.796)")

In [ ]:
# --- scikit-learn equivalent, so the notebook produces numbers without TensorFlow. ---
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

clf = MLPClassifier(hidden_layer_sizes=(11, 11), activation="relu", solver="adam",
                    max_iter=400, random_state=0, early_stopping=True,
                    validation_fraction=0.2)
clf.fit(X_train_s, y_train)
proba = clf.predict_proba(X_test_s)[:, 1]

baseline = (y_test == 0).mean()
print(f"majority-class baseline : {baseline:.4f}")
print(f"network test accuracy   : {accuracy_score(y_test, proba > 0.5):.4f}")

## Now do not stop at accuracy

A single number cannot tell you whether the model learned the minority class or simply
learned to say "no". The confusion matrix can.

In [ ]:
cm = confusion_matrix(y_test, proba > 0.5)
print("                predicted stay  predicted leave")
print(f"actually stay  {cm[0, 0]:>14}{cm[0, 1]:>17}")
print(f"actually leave {cm[1, 0]:>14}{cm[1, 1]:>17}")
print()
print(classification_report(y_test, proba > 0.5, target_names=["stayed", "left"], digits=3))

Recall on the "left" class is the number the bank actually cares about — of the customers
who really did leave, how many did we flag? It is well below the headline accuracy, and no
amount of staring at accuracy would have revealed that.

## The threshold is a business decision, not a default

`predict` gives a **probability**. Turning it into a yes/no needs a cut-off, and 0.5 is
just a convention. If a retention offer costs £20 and a lost customer costs £500, you
should be flagging far more aggressively than 0.5.

In [ ]:
print(f"{'threshold':>11}{'flagged':>9}{'caught':>8}{'recall':>9}{'precision':>11}")
for t in (0.2, 0.3, 0.4, 0.5, 0.6, 0.7):
    pred = proba > t
    caught = int((pred & (y_test == 1)).sum())
    total_left = int((y_test == 1).sum())
    prec = caught / max(pred.sum(), 1)
    print(f"{t:>11.1f}{int(pred.sum()):>9}{caught:>8}{caught / total_left:>9.3f}{prec:>11.3f}")
print(f"\n{int((y_test == 1).sum())} customers in the test set actually left")

Same model, same weights, entirely different behaviour. **Nothing was retrained** — only
the number the probability is compared against.

## Diagnosing with the loss curves

If you ran the Keras cell, `history.history` holds the training and validation loss per
epoch. The shape of those two curves is the whole diagnosis:

- both falling, close together → still learning, keep going
- training falling, validation rising → **overfitting**; add dropout, add regularisation,
  or stop earlier
- both flat and high → underfitting, or the learning rate is wrong

```python
import matplotlib.pyplot as plt
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="validation")
plt.xlabel("epoch"); plt.ylabel("binary cross-entropy"); plt.legend()
```

## Try it yourself

1. Retrain without scaling (`X_train` rather than `X_train_s`). How much accuracy do you
   lose, and does the model collapse to the majority class?
2. Set `drop_first=False` and retrain. Does the redundant column hurt? (It usually does
   not hurt *accuracy* — the interesting question is why people insist on dropping it.)
3. Pass `class_weight={0: 1, 1: 4}` to `MLPClassifier`-equivalent training in Keras. What
   happens to recall on "left", and what happens to precision?
4. Find the threshold that maximises £ saved under the £20 / £500 assumption above. It will
   not be 0.5.